# Single Store Team — Kaggle CPU
Accelerator=None, Internet=On. Use a fresh session. This notebook starts MySQL/API locally, never your personal database.
Commit/push the tested source first. Enter its SHA below; full four-phase run by default.

In [ ]:
from pathlib import Path
import re
import subprocess
import sys

repo = Path('/kaggle/working/store_ktpm')
ref = input('Commit SHA to benchmark (7-40 hex characters): ').strip()
if not re.fullmatch(r'[0-9a-fA-F]{7,40}', ref):
    raise ValueError('Use an actual pushed commit SHA, not a branch name or placeholder')
if repo.exists():
    raise RuntimeError('Repository already exists; use a fresh Kaggle session')
url = 'https:' + '//github.com/khoizz247/store_ktpm.git'
subprocess.run(['git', 'clone', url, str(repo)], check=True)
subprocess.run(['git', '-C', str(repo), 'checkout', '--detach', ref], check=True)
if not (repo / 'performance/kaggle_run.py').is_file():
    raise RuntimeError('This commit does not include the load runner')
print('Commit:', subprocess.check_output(['git', '-C', str(repo), 'rev-parse', 'HEAD'], text=True).strip())


In [ ]:
import os
env = os.environ.copy()
env.pop('LOAD_SMOKE_ONLY', None)
# For feasibility only: env['LOAD_SMOKE_ONLY'] = '1'
subprocess.run([sys.executable, str(repo / 'performance/kaggle_run.py')], cwd=repo, env=env, check=True)


In [ ]:
import shutil
runs = sorted(Path('/kaggle/working/single_store_load').glob('*'))
if not runs:
    raise RuntimeError('No evidence directory; inspect the runner error above')
latest = runs[-1]
report = latest / 'REPORT.md'
print(report.read_text(encoding='utf-8') if report.exists() else 'No completed report: inspect api.log/runner output')
archive = shutil.make_archive('/kaggle/working/single_store_load_results', 'zip', root_dir='/kaggle/working', base_dir='single_store_load')
print('Download from Kaggle Output:', archive)
